## APIs: collect weather data

Weather is a huge factor when it comes to scooter usage. On very cold days and, especially, on rainy days, hardly anyone wants to use a scooter.

Thankfully, there is an API that provides free weather forecasts: [OpenWeather](https://openweathermap.org/api).

If you followed our guidance, you already have created an account. If you forgot, go back to the LMS section 🔧Setting up accounts for the project.

OpenWeather has many APIs that you can use. Once your API key arrives, we recommend using the [5 day forecast](https://openweathermap.org/forecast5). But if you feel confident and wish to explore, you are of course free to choose another one of the APIs.

It’s easy to make an API call using code similar to below:

```python
f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API key}&units=metric"
```

Obviously, latitude, longitude, and API_key are variables that you will have already defined. Once you’ve made this API call, you’ll be able to explore the received information using the [requests](https://docs.python-requests.org/en/latest/) library and pandas.

### 🧩 The goal for today is to automate the collection of weather data for multiple cities. 

1. Explore the data received from the API. What’s useful? What’s not?
2. Extract the information you see as useful and make a DataFrame from it
3. Consolidate the steps you took to make the DataFrame into a function. The function should output a DataFrame with the weather of multiple cities, when given a list of cities as an input. If you’re still unsure about functions, here’s a guide.
4. You should be able to use this function to get the weather data for the cities you web scraped yesterday



In [221]:
import os 
from dotenv import load_dotenv
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
import pymysql

load_dotenv()

connection_string = f'mysql+pymysql://{os.environ["MYSQL_USER"]}:{os.environ["MYSQL_ROOT_PASSWORD"]}@{os.environ["MYSQL_HOST"]}:{os.environ["MYSQL_PORT"]}/gans' #/{schema}'
api_key = os.environ["API_KEY"]

In [222]:
def get_weather(lat, lon):
    return requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&units=metric")

In [223]:
def city_loc(city_df):
    forecast = {
        "city": [],
        "weather": [],
        "short": [],
    }
    for _, i in city_df.iterrows():
        weather = get_weather(i["latitude"], i["longitude"]).json()
        
        forecast["city"].append(pd.json_normalize(weather["city"]))
        forecast["city"][-1]["city_name"] = i["name"]
        forecast["city"][-1]["city_id"] = i["city_id"]
        forecast["city"][-1]["country_name"] = i["country"]
        
        fc = pd.json_normalize(weather["list"]).drop("weather", axis=1)
        fc["city_id"] = weather["city"]["id"]
        fc["city_id_sql"] = i["city_id"]
        forecast["weather"].append(fc)
    
        short = pd.json_normalize(
            weather["list"],
            record_path="weather",   # explode each row's list of dicts
            meta=["dt"],             # parent fields to attach to every exploded row
        )
        short["city_id"] = weather["city"]["id"]
        forecast["short"].append(short)

    for k, i in forecast.items():
        forecast[k] = pd.concat(i, ignore_index=True)

    return forecast

In [224]:
df = pd.read_sql(
    "SELECT * FROM city",
    con = connection_string,
)

In [225]:
df

,city_id,name,country,country_code,timezone,population,latitude,longitude
0,1,Berlin,Germany,None,None,3596999,52.518667,13.403000
1,2,Hamburg,Germany,None,None,1973896,53.550000,10.000000
2,3,Munich,Germany,None,None,1505036,48.135833,11.571667


In [226]:
forecast = city_loc(df)

In [227]:
forecast["city"]

,id,name,country,population,timezone,sunrise,sunset,coord.lat,coord.lon,city_name,city_id,country_name
0,6545310,Mitte,DE,329078,7200,1792561318,1792598388,52.5200,13.4050,Berlin,1,Germany
1,2911298,Hamburg,DE,1739117,7200,1792562260,1792599080,53.5500,10.0000,Hamburg,2,Germany
2,2867714,Munich,DE,1260391,7200,1792561291,1792599294,48.1358,11.5717,Munich,3,Germany


In [228]:
if not "rain.3h" in forecast["weather"].columns:
    forecast["weather"]["rain.3h"] = 0

In [229]:
forecast["weather"]

,dt,visibility,pop,dt_txt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,main.sea_level,...,main.temp_kf,main.dew_point,clouds.all,wind.speed,wind.deg,wind.gust,sys.pod,rain.3h,city_id,city_id_sql
0,1792584000,10000,0.00,2026-10-21 12:00:00,13.02,12.54,13.02,13.60,1009,1009,...,-0.58,10.20,98,1.84,95,4.64,d,NaN,6545310,1
1,1792594800,10000,0.00,2026-10-21 15:00:00,13.40,12.91,13.40,14.17,1009,1009,...,-0.77,10.21,97,1.29,129,2.35,d,NaN,6545310,1
2,1792605600,10000,0.00,2026-10-21 18:00:00,11.74,11.22,11.10,11.74,1008,1008,...,0.64,9.48,95,1.95,192,3.74,n,NaN,6545310,1
3,1792616400,6160,0.70,2026-10-21 21:00:00,11.23,10.86,11.23,11.23,1009,1009,...,0.00,8.29,100,2.82,247,5.59,n,NaN,6545310,1
4,1792627200,3139,1.00,2026-10-22 00:00:00,9.27,7.43,9.27,9.27,1011,1011,...,0.00,8.54,100,3.35,257,6.33,n,4.33,6545310,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1792962000,9429,0.47,2026-10-25 21:00:00,7.76,4.59,7.76,7.76,1017,1017,...,0.00,5.45,37,5.48,268,11.26,n,NaN,2867714,3
116,1792972800,10000,0.20,2026-10-26 00:00:00,6.96,4.00,6.96,6.96,1020,1020,...,0.00,4.97,16,4.54,265,9.24,n,0.27,2867714,3
117,1792983600,10000,0.00,2026-10-26 03:00:00,6.20,3.28,6.20,6.20,1021,1021,...,0.00,4.60,9,4.09,251,8.25,n,NaN,2867714,3
118,1792994400,10000,0.00,2026-10-26 06:00:00,5.49,2.76,5.49,5.49,1023,1023,...,0.00,4.25,61,3.50,246,6.78,d,NaN,2867714,3


In [230]:
forecast["short"]

,id,main,description,icon,dt,city_id
0,804,Clouds,overcast clouds,04d,1792584000,6545310
1,804,Clouds,overcast clouds,04d,1792594800,6545310
2,804,Clouds,overcast clouds,04n,1792605600,6545310
3,804,Clouds,overcast clouds,04n,1792616400,6545310
4,501,Rain,moderate rain,10n,1792627200,6545310
...,...,...,...,...,...,...
115,802,Clouds,scattered clouds,03n,1792962000,2867714
116,500,Rain,light rain,10n,1792972800,2867714
117,800,Clear,clear sky,01n,1792983600,2867714
118,803,Clouds,broken clouds,04d,1792994400,2867714


### Compose data for database

#### Constant Information

- city_id
- city_name
- country
- country_code
- timezone
- latitude
- longitude
- population

#### Annual Information

Normally the population, but for the purpose of this project, the population is considered constant.

#### Daily Information

- sunrise
- sunset

#### 3h Information

- dt
- dt_txt
- visibility
- probability of precipitation (pop)
- main.temp
- main.feels_like
- main.pressure
- main.humidity
- wind.speed
- wind.deg
- wind.gust
- sys.pod (part of day)
- rain.3h
- weather_id
- weather.main
- description
- icon
- dt
- city_id

```sql
CREATE TABLE city (
    city_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(255) NOT NULL,
    country VARCHAR(255) NOT NULL,
    country_code CHAR(2),
    timezone INT,
    population INT,
    latitude DECIMAL(10 , 7 ),
    longitude DECIMAL(10 , 7 )
);

CREATE TABLE solar_time (
    city_id INT NOT NULL,
    sunrise INT NOT NULL,
    sunset INT NOT NULL,
    sunrise_text TEXT,
    sunset_text TEXT,
    FOREIGN KEY (city_id)
        REFERENCES city (city_id),
	UNIQUE (city_id, sunrise)
);

CREATE TABLE forecast (
	city_id INT NOT NULL,
    `time` INT NOT NULL,
    time_text TEXT,
    temperature DECIMAL(5, 2),
    temperature_felt DECIMAL(5, 2),
    weather TEXT,
    `description` TEXT,
    weather_id TEXT,
    precipitation_prob INT,
    rain_3h DECIMAL(5, 2),
    clouds INT,
    pressure INT,
    humidity INT,
    icon_id CHAR(3),
    FOREIGN KEY (city_id)
        REFERENCES city (city_id),
	UNIQUE (city_id, `time`)
);
```

In [231]:
forecast["city"]

,id,name,country,population,timezone,sunrise,sunset,coord.lat,coord.lon,city_name,city_id,country_name
0,6545310,Mitte,DE,329078,7200,1792561318,1792598388,52.5200,13.4050,Berlin,1,Germany
1,2911298,Hamburg,DE,1739117,7200,1792562260,1792599080,53.5500,10.0000,Hamburg,2,Germany
2,2867714,Munich,DE,1260391,7200,1792561291,1792599294,48.1358,11.5717,Munich,3,Germany


In [232]:
city_update = forecast["city"].loc[:, ["city_id", "country", "timezone"]]

In [233]:
city_update = city_update.rename({"country": "country_code"}, axis=1)

In [234]:
solar_time = forecast["city"].loc[:, ["city_id", "sunrise", "sunset"]]

In [235]:
solar_time["sunrise_text"] = pd.to_datetime(solar_time.loc[:, "sunrise"] + forecast["city"].loc[:, "timezone"], unit="s").dt.strftime("%Y-%m-%d %H-%M-%S")
solar_time["sunset_text"] = pd.to_datetime(solar_time.loc[:, "sunset"] + forecast["city"].loc[:, "timezone"], unit="s").dt.strftime("%Y-%m-%d %H-%M-%S")

In [236]:
solar_time

,city_id,sunrise,sunset,sunrise_text,sunset_text
0,1,1792561318,1792598388,2026-10-21 07-41-58,2026-10-21 17-59-48
1,2,1792562260,1792599080,2026-10-21 07-57-40,2026-10-21 18-11-20
2,3,1792561291,1792599294,2026-10-21 07-41-31,2026-10-21 18-14-54


In [237]:
forecast["weather"]

,dt,visibility,pop,dt_txt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,main.sea_level,...,main.temp_kf,main.dew_point,clouds.all,wind.speed,wind.deg,wind.gust,sys.pod,rain.3h,city_id,city_id_sql
0,1792584000,10000,0.00,2026-10-21 12:00:00,13.02,12.54,13.02,13.60,1009,1009,...,-0.58,10.20,98,1.84,95,4.64,d,NaN,6545310,1
1,1792594800,10000,0.00,2026-10-21 15:00:00,13.40,12.91,13.40,14.17,1009,1009,...,-0.77,10.21,97,1.29,129,2.35,d,NaN,6545310,1
2,1792605600,10000,0.00,2026-10-21 18:00:00,11.74,11.22,11.10,11.74,1008,1008,...,0.64,9.48,95,1.95,192,3.74,n,NaN,6545310,1
3,1792616400,6160,0.70,2026-10-21 21:00:00,11.23,10.86,11.23,11.23,1009,1009,...,0.00,8.29,100,2.82,247,5.59,n,NaN,6545310,1
4,1792627200,3139,1.00,2026-10-22 00:00:00,9.27,7.43,9.27,9.27,1011,1011,...,0.00,8.54,100,3.35,257,6.33,n,4.33,6545310,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1792962000,9429,0.47,2026-10-25 21:00:00,7.76,4.59,7.76,7.76,1017,1017,...,0.00,5.45,37,5.48,268,11.26,n,NaN,2867714,3
116,1792972800,10000,0.20,2026-10-26 00:00:00,6.96,4.00,6.96,6.96,1020,1020,...,0.00,4.97,16,4.54,265,9.24,n,0.27,2867714,3
117,1792983600,10000,0.00,2026-10-26 03:00:00,6.20,3.28,6.20,6.20,1021,1021,...,0.00,4.60,9,4.09,251,8.25,n,NaN,2867714,3
118,1792994400,10000,0.00,2026-10-26 06:00:00,5.49,2.76,5.49,5.49,1023,1023,...,0.00,4.25,61,3.50,246,6.78,d,NaN,2867714,3


In [240]:
forecast_table = (
    forecast["weather"]
        .set_index(["dt", "city_id"])
        .join(
            forecast["short"].set_index(["dt", "city_id"]),
            how="inner"
        )
        .reset_index()
        .drop_duplicates(["dt", "city_id"])
        .loc[:, [
            "city_id_sql",
            "dt",
            "dt_txt",
            "main.temp",
            "main.feels_like",
            "main",
            "description",
            "id",
            "rain.3h",
            "clouds.all",
            "main.pressure",
            "main.humidity",
            "icon",
        ]]
        .rename({
            "dt": "time",
            "dt_txt": "time_text",
            "main.temp": "temperature",
            "main.feels_like": "temperature_felt",
            "main": "weather",
            "id": "weather_id",
            "rain.3h": "rain_3h",
            "clouds.all": "clouds",
            "main.pressure": "pressure",
            "main.humidity": "humidity",
            "icon": "icon_id",
            "city_id_sql": "city_id",
        }, axis=1)
)
forecast_table["precipitation_prob"] = (forecast["weather"].loc[:, "pop"] * 100).astype(int)

In [241]:
forecast_table

,city_id,time,time_text,temperature,temperature_felt,weather,description,weather_id,rain_3h,clouds,pressure,humidity,icon_id,precipitation_prob
0,1,1792584000,2026-10-21 12:00:00,13.02,12.54,Clouds,overcast clouds,804,NaN,98,1009,83,04d,0
1,1,1792594800,2026-10-21 15:00:00,13.40,12.91,Clouds,overcast clouds,804,NaN,97,1009,81,04d,0
2,1,1792605600,2026-10-21 18:00:00,11.74,11.22,Clouds,overcast clouds,804,NaN,95,1008,86,04n,0
3,1,1792616400,2026-10-21 21:00:00,11.23,10.86,Clouds,overcast clouds,804,NaN,100,1009,94,04n,70
4,1,1792627200,2026-10-22 00:00:00,9.27,7.43,Rain,moderate rain,501,4.33,100,1011,95,10n,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,3,1792962000,2026-10-25 21:00:00,7.76,4.59,Clouds,scattered clouds,802,NaN,37,1017,86,03n,47
116,3,1792972800,2026-10-26 00:00:00,6.96,4.00,Rain,light rain,500,0.27,16,1020,88,10n,20
117,3,1792983600,2026-10-26 03:00:00,6.20,3.28,Clear,clear sky,800,NaN,9,1021,90,01n,0
118,3,1792994400,2026-10-26 06:00:00,5.49,2.76,Clouds,broken clouds,803,NaN,61,1023,93,04d,0


In [242]:
city_update.to_sql(
    'city_tmp',
    schema="gans",
    if_exists='replace',
    con=connection_string,
    index=False,
)

3

In [243]:
solar_time.to_sql(
    'solar_time',
    schema="gans",
    if_exists='append',
    con=connection_string,
    index=False,
)

3

In [244]:
forecast_table.to_sql(
    'forecast',
    schema="gans",
    if_exists='append',
    con=connection_string,
    index=False,
)

120

In [ ]:
conn = pymysql.connect(
    host=os.environ["MYSQL_HOST"],
    port=int(os.environ["MYSQL_PORT"]),
    user=os.environ["MYSQL_USER"],
    password=os.environ["MYSQL_ROOT_PASSWORD"],
    database="gans",
)

with conn.cursor() as c:
    c.callproc("update_city")

conn.commit()
conn.close()